# 02 Transformer 前向传播与因果性
这里创建一个很小的随机模型验证结构。随机输出没有语言意义。

In [ ]:
from pathlib import Path
import sys, torch
ROOT = Path.cwd()
if not (ROOT / 'model').exists(): ROOT = ROOT.parents[2]
sys.path.insert(0, str(ROOT))
from model.config import MiniLLMConfig
from model.gpt import MiniLLM

In [ ]:
cfg = MiniLLMConfig(vocab_size=128, block_size=32, n_layer=2, n_head=4, n_embd=64, intermediate_size=128)
model = MiniLLM(cfg).eval()
x = torch.randint(0, cfg.vocab_size, (2, 12))
out = model(x)['logits']
print('input:', x.shape, 'logits:', out.shape)
print('parameters:', model.num_params())

因果性测试：修改位置 7 之后的 token，不应改变位置 0–6 的 logits。

In [ ]:
x2 = x.clone()
x2[:, 7:] = torch.randint(0, cfg.vocab_size, x2[:, 7:].shape)
with torch.no_grad():
    y1 = model(x)['logits']
    y2 = model(x2)['logits']
past_error = (y1[:, :7] - y2[:, :7]).abs().max().item()
future_error = (y1[:, 7:] - y2[:, 7:]).abs().max().item()
print('past max error:', past_error)
print('changed region error:', future_error)
assert past_error < 1e-6

In [ ]:
full_cfg = MiniLLMConfig.from_yaml(str(ROOT / 'configs/model_config.yaml'))
full_model = MiniLLM(full_cfg)
print(f'正式模型参数量: {full_model.num_params():,}')
del full_model